# Blocco 3: Pipeline Completa

---

## La scena

Sono le 9:00 di lunedi mattina. Il Dott. Marco Ferretti ti chiama nel suo ufficio.

> *"Complimenti per il lavoro sui dati e per la padronanza di xlwings. Sei promosso Assistant Advisor.  
> Ma ora viene la parte importante: devi **automatizzare tutto**.  
> Il report trimestrale per la Famiglia Bianchi deve generarsi con un click.  
> Multi-foglio, protetto, esportabile in PDF. Niente scuse."*

---

## Cosa impariamo in questo blocco

| Esercizio | Argomento | Competenza |
|-----------|-----------|------------|
| **ES10** | Script completo | Organizzazione in funzioni riutilizzabili |
| **ES11** | Protezione e consegna | Protezione fogli, formule nascoste |
| **ES12** | Report trimestrale Bianchi | Report multi-foglio professionale |
| **ES13** | Esporta PDF | Layout di stampa, export PDF |
| **ES14** | Il confronto finale | Simulazione vs realta |

---

**Prerequisiti**: Blocco 1 (pandas, yfinance) e Blocco 2 (xlwings, formattazione, formule) completati.

In [ ]:
# ============================================================
# SETUP COMPLETO - esegui questa cella prima di tutto
# ============================================================
import xlwings as xw
import pandas as pd
import numpy as np
import yfinance as yf
import sys, os
from datetime import datetime

# Aggiungi la cartella scripts al path Python
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("")), "scripts"))

# Importa configurazioni Turin Wealth
from tw_config import COLORS, AZIENDE, BENCHMARK, NUMBER_FORMATS, CLIENTE

# Importa utilities xlwings
from tw_utils import (
    set_formula, fmt_header, fmt_title, protect_sheet,
    create_workbook, save_and_close, write_table, add_sheet,
    hide_sheet, autofit_all
)

# Cartelle di lavoro
CACHE_DIR = os.path.join(os.path.dirname(os.path.abspath("")), "lezione", "dati_cache")
OUTPUT_DIR = os.path.join(os.path.dirname(os.path.abspath("")), "lezione")

# Crea cartella cache se non esiste
os.makedirs(CACHE_DIR, exist_ok=True)


def scarica_o_cache(ticker, period="5y"):
    """
    Scarica dati da Yahoo Finance o usa la cache locale.
    
    Strategia:
    1. Prova a scaricare da yfinance
    2. Se fallisce (no internet, rate limit), carica da file CSV locale
    3. Se nemmeno la cache esiste, lancia FileNotFoundError
    """
    try:
        df = yf.download(ticker, period=period, progress=False)
        if len(df) > 0:
            # Salva in cache per usi futuri
            nome_file = ticker.replace("^", "").replace(".", "_") + ".csv"
            path_cache = os.path.join(CACHE_DIR, nome_file)
            df.to_csv(path_cache)
            return df
    except Exception:
        pass

    # Fallback: cache locale
    nome_file = ticker.replace("^", "").replace(".", "_") + ".csv"
    path_cache = os.path.join(CACHE_DIR, nome_file)
    if os.path.exists(path_cache):
        print(f"  [cache] Uso dati locali per {ticker}")
        return pd.read_csv(path_cache, index_col=0, parse_dates=True)

    raise FileNotFoundError(f"Nessun dato disponibile per {ticker} (ne online ne in cache)")


print("Setup completato.")
print(f"Cache dir: {CACHE_DIR}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Aziende configurate: {[az['nome'] for az in AZIENDE]}")
print(f"Benchmark configurati: {[b['nome'] for b in BENCHMARK]}")

---
## ES10: Lo script completo

### Il problema del codice "usa e getta"

Finora abbiamo scritto codice in modo lineare: prima scarica, poi elabora, poi scrivi. Funziona per un test, ma non per un report che si rigenera ogni trimestre.

**La soluzione: organizzare il codice in funzioni riutilizzabili.**

Ogni funzione fa **una sola cosa** (principio di responsabilita singola):
- `scarica_dati_aziende()` → solo lo scaricamento
- `calcola_statistiche()` → solo i calcoli
- `crea_report_excel()` → solo la scrittura su Excel

Cosi posso:
- Testare ogni parte separatamente
- Aggiornare una parte senza rompere le altre
- Riusare le funzioni in altri script

In [ ]:
# ============================================================
# ES10 - DEMO: Funzioni modulari
# ============================================================

def scarica_dati_aziende():
    """
    Scarica prezzi Close per le 5 aziende Turin Wealth.
    
    Ritorna: DataFrame con colonne = nomi aziende, indice = date
    """
    dati = {}
    for az in AZIENDE:
        df = scarica_o_cache(az["ticker"])
        # Gestisci MultiIndex colonne (yfinance v0.2+)
        if isinstance(df.columns, pd.MultiIndex):
            dati[az["nome"]] = df["Close"][az["ticker"]]
        else:
            dati[az["nome"]] = df["Close"]
        print(f"  {az['nome']}: {len(df)} giorni di dati")
    return pd.DataFrame(dati)


def calcola_statistiche(prezzi):
    """
    Calcola statistiche di performance per ogni titolo.
    
    Input:  DataFrame prezzi (date x titoli)
    Output: DataFrame statistiche (titoli x metriche)
    """
    rendimenti = prezzi.pct_change().dropna()
    
    stats = pd.DataFrame({
        "Ultimo prezzo":     prezzi.iloc[-1],
        "Rend. 1M (%)":      ((prezzi.iloc[-1] / prezzi.iloc[-22]) - 1) * 100,
        "Rend. 1Y (%)":      ((prezzi.iloc[-1] / prezzi.iloc[-252]) - 1) * 100,
        "Volatilita ann. (%)": rendimenti.std() * np.sqrt(252) * 100,
        "Sharpe":            (rendimenti.mean() * 252) / (rendimenti.std() * np.sqrt(252)),
    }).round(2)
    
    return stats


def crea_report_excel(prezzi, stats, output_path):
    """
    Crea il report Excel completo.
    (Scheletro - lo completiamo negli esercizi successivi)
    """
    wb = create_workbook(visible=False)
    # ... implementazione nei prossimi esercizi
    return wb


# ---- Test delle funzioni ----
print("Scaricamento dati aziende...")
prezzi = scarica_dati_aziende()

print(f"\nDataFrame prezzi: {prezzi.shape[0]} righe x {prezzi.shape[1]} colonne")
print(f"Periodo: {prezzi.index[0].date()} -> {prezzi.index[-1].date()}")

stats = calcola_statistiche(prezzi)
print("\nStatistiche calcolate:")
stats

### Composizione di funzioni

Le tre funzioni si compongono in una pipeline lineare:

```
scarica_dati_aziende()  →  prezzi (DataFrame)
        ↓
calcola_statistiche(prezzi)  →  stats (DataFrame)
        ↓
crea_report_excel(prezzi, stats, path)  →  file .xlsx
```

In uno script Python standalone (`create_report.py`) useresti il pattern `if __name__ == "__main__":` per eseguire la pipeline solo quando lo script viene lanciato direttamente:

```python
if __name__ == "__main__":
    prezzi = scarica_dati_aziende()
    stats = calcola_statistiche(prezzi)
    crea_report_excel(prezzi, stats, "Report_Q1.xlsx")
    print("Report generato con successo!")
```

Nel notebook questo non serve: basta eseguire le celle in ordine.

In [ ]:
# ============================================================
# ES10 - ESERCIZIO: Funzione scarica_benchmark()
# ============================================================
# Scrivi la funzione scarica_benchmark() seguendo lo stesso
# pattern di scarica_dati_aziende().
#
# Suggerimento:
#   - BENCHMARK e' una lista di dizionari con chiavi 'nome' e 'ticker'
#   - Ogni benchmark ha un ticker Yahoo Finance (es. '^GSPC' per S&P 500)
#   - Usa scarica_o_cache() per ogni ticker
#   - Ritorna un DataFrame con colonne = nomi benchmark, indice = date
#
# Nota: i ticker con '^' (es. '^GSPC') vengono rinominati da scarica_o_cache
# per il salvataggio su file. Controlla come viene costruito nome_file.

def scarica_benchmark():
    """Scarica i dati degli indici benchmark."""
    # ???
    # Suggerimento: struttura uguale a scarica_dati_aziende()
    # ma itera su BENCHMARK invece di AZIENDE
    pass


# Testa la funzione (decommenta quando pronta)
# bench = scarica_benchmark()
# print(f"Benchmark: {list(bench.columns)}")
# bench.tail(3)

In [ ]:
# ============================================================
# ES10 - SOLUZIONE
# ============================================================

def scarica_benchmark():
    """Scarica i dati degli indici benchmark."""
    dati = {}
    for b in BENCHMARK:
        try:
            df = scarica_o_cache(b["ticker"])
            # Gestisci MultiIndex colonne (yfinance v0.2+)
            if isinstance(df.columns, pd.MultiIndex):
                dati[b["nome"]] = df["Close"][b["ticker"]]
            else:
                dati[b["nome"]] = df["Close"]
            print(f"  {b['nome']}: {len(df)} giorni di dati")
        except FileNotFoundError as e:
            print(f"  [SKIP] {b['nome']}: {e}")
    return pd.DataFrame(dati)


print("Scaricamento benchmark...")
bench = scarica_benchmark()
print(f"\nBenchmark disponibili: {list(bench.columns)}")
bench.tail(3)

---
## ES11: Protezione e consegna

> *"Il file deve essere protetto. I clienti non devono rompere le formule.  
> E voglio che le formule di verifica non siano visibili nella barra in alto."*  
> — Dott. Ferretti

### Gerarchia di protezione in Excel

Excel ha tre livelli di protezione che funzionano insieme:

```
1. Cella → api.Locked = True/False
             (default: True per tutte le celle)
        ↓
2. Cella → api.FormulaHidden = True
             (nasconde formula nella barra)
        ↓
3. Foglio → protect_sheet(ws, password)
             (ATTIVA i flag Locked e FormulaHidden)
```

**Importante**: `Locked` e `FormulaHidden` non fanno NIENTE finche il foglio non e protetto. E' il terzo passaggio che li attiva.

In [ ]:
# ============================================================
# ES11 - DEMO: Protezione foglio
# ============================================================

# Crea un workbook di esempio
app = xw.App(visible=True)
wb_demo = app.books.add()
ws = wb_demo.sheets[0]
ws.name = "Report"

# --- Intestazione ---
ws.range("A1").value = "REPORT TRIMESTRALE - FAMIGLIA BIANCHI"
fmt_title(ws.range("A1:D1"))
ws.range("A1:D1").merge()

# --- Tabella dati ---
ws.range("A3").value = [["Titolo", "Prezzo", "Variazione %"]]
fmt_header(ws.range("A3:C3"))

ws.range("A4").value = [
    ["Terna",     7.85,   0.032],
    ["Ferrari",   420.50, 0.125],
    ["Microsoft", 425.00, 0.087],
]

# --- Formula nascosta ---
# Calcola variazione media (formula che non vogliamo mostrare)
set_formula(ws.range("D3"), "=MEDIA(C4:C6)")
ws.range("D3").value = "Var. media"  # sostituisci con etichetta
set_formula(ws.range("D4"), "=MEDIA(C4:C6)")
ws.range("D4").api.FormulaHidden = True  # nasconde nella barra formule

print("Prima della protezione:")
print(f"  FormulaHidden attivo? {ws.range('D4').api.FormulaHidden}")
print("  (clicca su D4 e vedi la formula nella barra sopra)")

# --- Sblocca celle modificabili dal cliente ---
# Prima sblocca TUTTE le celle che il cliente puo toccare
ws.range("C4:C6").api.Locked = False   # prezzi aggiornabili
print("\nCelle C4:C6 sbloccate (il cliente potra modificarle)")

# --- Proteggi il foglio ---
protect_sheet(ws, "password123")
print("\nFoglio protetto!")
print("  - Celle A1:B6 e D4: bloccate (non modificabili)")
print("  - Celle C4:C6: sbloccate (il cliente puo aggiornare i prezzi)")
print("  - Formula in D4: nascosta nella barra formule")
print("\nProva a cliccare sulle celle grigie: Excel mostrera un errore.")

### Fogli nascosti: `very_hidden`

Nei file di esercitazione Turin Wealth, i fogli `_CORREZIONE` contengono le risposte corrette. Questi fogli devono essere **completamente invisibili**: non solo nascosti, ma nascondibili solo via VBA/Python, non dall'interfaccia Excel.

```python
# Nasconde nella finestra Formato > Nascondi foglio  
ws.api.Visible = 0              # xlSheetHidden - visibile da Format > Unhide

# Nasconde completamente: non appare nemmeno in Format > Unhide
ws.api.Visible = 2              # xlSheetVeryHidden - solo via VBA/Python
```

La funzione `hide_sheet(ws, very_hidden=True)` di `tw_utils.py` gestisce questo automaticamente.

In [ ]:
# ============================================================
# ES11 - DEMO: Foglio very_hidden
# ============================================================

# Aggiungi un foglio segreto
ws_segreto = wb_demo.sheets.add("_CORREZIONE")
ws_segreto.range("A1").value = "Risposte corrette - solo docente!"
ws_segreto.range("A2").value = "Terna rendimento: +3.2%"
ws_segreto.range("A3").value = "Ferrari rendimento: +12.5%"

# Nascondi con very_hidden
hide_sheet(ws_segreto, very_hidden=True)

# Mostra quali fogli sono visibili
fogli_visibili = [s.name for s in wb_demo.sheets if s.api.Visible == -1]  # xlSheetVisible = -1
fogli_nascosti = [s.name for s in wb_demo.sheets if s.api.Visible != -1]

print(f"Fogli visibili in Excel: {fogli_visibili}")
print(f"Fogli nascosti (very hidden): {fogli_nascosti}")
print("\nProva in Excel: Home > Formato > Nascondi e mostra > Mostra foglio")
print("Il foglio '_CORREZIONE' non appare nell'elenco!")

In [ ]:
# ============================================================
# ES11 - ESERCIZIO: Protezione avanzata
# ============================================================
# Aggiungi un nuovo foglio "Budget" al workbook wb_demo e configuralo:
#
# 1. Scrivi in A1: "BUDGET TRIMESTRALE"
# 2. Scrivi in A3:B3 gli header: ["Voce", "Importo"]
# 3. Scrivi in A4:B6 i dati:
#      ["Commissioni advisory", 14000]
#      ["Spese di custodia",     2800]
#      ["Altri costi",           1200]
# 4. Inserisci in B7 una formula che somma B4:B6
#    Nascondi la formula con FormulaHidden = True
# 5. Sblocca SOLO le celle B4:B6 (importi modificabili)
# 6. Proteggi il foglio con password "turin2024"
# 7. Verifica: clicca su A1 e su B4 in Excel. Quale puoi modificare?

# ???

# Atteso:
# - ws_budget.range("A1").api.Locked == True  (non modificabile)
# - ws_budget.range("B4").api.Locked == False (modificabile)
# - ws_budget.range("B7").api.FormulaHidden == True

In [ ]:
# ============================================================
# ES11 - SOLUZIONE
# ============================================================

ws_budget = wb_demo.sheets.add("Budget")

# 1. Titolo
ws_budget.range("A1").value = "BUDGET TRIMESTRALE"
fmt_title(ws_budget.range("A1:B1"))
ws_budget.range("A1:B1").merge()

# 2. Header
ws_budget.range("A3").value = [["Voce", "Importo"]]
fmt_header(ws_budget.range("A3:B3"))

# 3. Dati
ws_budget.range("A4").value = [
    ["Commissioni advisory", 14000],
    ["Spese di custodia",     2800],
    ["Altri costi",           1200],
]

# 4. Formula totale con formula nascosta
ws_budget.range("A7").value = "TOTALE"
ws_budget.range("A7").font.bold = True
set_formula(ws_budget.range("B7"), "=SOMMA(B4:B6)")
ws_budget.range("B7").api.FormulaHidden = True   # attivo solo dopo protect
ws_budget.range("B7").api.NumberFormatLocal = "#.##0 \u20ac"

# 5. Sblocca solo le celle importo
ws_budget.range("B4:B6").api.Locked = False
ws_budget.range("B4:B6").api.NumberFormatLocal = "#.##0 \u20ac"

# 6. Proteggi
protect_sheet(ws_budget, "turin2024")

print("Foglio Budget creato e protetto!")
print(f"  Locked A1: {ws_budget.range('A1').api.Locked}  (atteso True)")
print(f"  Locked B4: {ws_budget.range('B4').api.Locked}  (atteso False)")
print(f"  FormulaHidden B7: {ws_budget.range('B7').api.FormulaHidden}  (atteso True)")

# Chiudi workbook demo
wb_demo.close()
app.quit()
print("\nWorkbook demo chiuso.")

---
## ES12: Report trimestrale Bianchi

> *"Il pezzo forte. Un report multi-foglio professionale che i Bianchi riceveranno ogni trimestre."*

### Architettura del report

Il report e strutturato in tre fogli:

| Foglio | Contenuto | Destinatario |
|--------|-----------|-------------|
| **Copertina** | Branding, dati cliente, data | Primo impatto visivo |
| **Statistiche** | Tabella metriche per titolo | Roberto e Laura |
| **Grafici** | Andamento prezzi ultimi 60 giorni | Tutti |

Ogni foglio e costruito da una sezione dedicata dentro `crea_report_bianchi()`.  
I dati fluiscono come parametri: `prezzi` e `stats` sono calcolati una volta e passati alla funzione.

In [ ]:
# ============================================================
# ES12 - DEMO: Report multi-foglio
# ============================================================

def crea_report_bianchi(prezzi, stats):
    """
    Genera il report trimestrale completo per la Famiglia Bianchi.
    
    Input:
        prezzi: DataFrame (date x titoli) - prezzi storici
        stats:  DataFrame (titoli x metriche) - statistiche calcolate
    Output:
        wb: workbook xlwings aperto (da salvare/chiudere chiamante)
    """
    wb = create_workbook(visible=True)

    # --------------------------------------------------------
    # Foglio 1: Copertina
    # --------------------------------------------------------
    ws_cop = wb.sheets[0]
    ws_cop.name = "Copertina"

    # Larghezze colonne per layout
    ws_cop.range("A:A").column_width = 4
    ws_cop.range("B:B").column_width = 28
    ws_cop.range("C:C").column_width = 20

    # Logo / Nome societa
    ws_cop.range("B3").value = "TURIN WEALTH ADVISORY"
    ws_cop.range("B3").font.size = 22
    ws_cop.range("B3").font.bold = True
    ws_cop.range("B3").font.color = COLORS["header"]

    ws_cop.range("B4").value = "Wealth Management & Advisory"
    ws_cop.range("B4").font.size = 13
    ws_cop.range("B4").font.italic = True
    ws_cop.range("B4").font.color = COLORS["subheader"]

    # Linea separatrice
    ws_cop.range("B6:D6").api.Borders(9).Weight = 2   # xlEdgeBottom
    ws_cop.range("B6:D6").api.Borders(9).Color = sum(
        v * (256 ** i) for i, v in enumerate(reversed(COLORS["accent"]))
    )

    # Titolo report
    ws_cop.range("B8").value = "Report Trimestrale"
    ws_cop.range("B8").font.size = 17
    ws_cop.range("B8").font.bold = True

    trimestre = f"Q{((datetime.now().month - 1) // 3) + 1} {datetime.now().year}"
    ws_cop.range("B9").value = trimestre
    ws_cop.range("B9").font.size = 14
    ws_cop.range("B9").font.color = COLORS["accent"]

    # Dati cliente
    ws_cop.range("B12").value = "Cliente:"
    ws_cop.range("B12").font.bold = True
    ws_cop.range("C12").value = CLIENTE["nome"]

    ws_cop.range("B13").value = "Data:"
    ws_cop.range("B13").font.bold = True
    ws_cop.range("C13").value = datetime.now().strftime("%d/%m/%Y")

    ws_cop.range("B14").value = "Consulente:"
    ws_cop.range("B14").font.bold = True
    ws_cop.range("C14").value = CLIENTE["advisor"]

    ws_cop.range("B15").value = "Patrimonio gestito:"
    ws_cop.range("B15").font.bold = True
    ws_cop.range("C15").value = CLIENTE["patrimonio_totale"]
    ws_cop.range("C15").api.NumberFormatLocal = "\u20ac #.##0"

    # --------------------------------------------------------
    # Foglio 2: Statistiche
    # --------------------------------------------------------
    ws_stats = add_sheet(wb, "Statistiche")

    # Titolo
    ws_stats.range("A1").value = "ANALISI TITOLI — " + trimestre
    fmt_title(ws_stats.range("A1:F1"))
    ws_stats.range("A1:F1").merge()
    ws_stats.row_height = 28

    # Tabella
    headers = list(stats.columns)
    data = [[idx] + list(row) for idx, row in stats.iterrows()]
    write_table(ws_stats, (2, 1), ["Titolo"] + headers, data)

    # Formato numeri
    n_rows = len(data)
    ws_stats.range(f"B3:B{2 + n_rows}").api.NumberFormatLocal = "#.##0,00 \u20ac"
    ws_stats.range(f"C3:E{2 + n_rows}").api.NumberFormatLocal = "0,00"
    ws_stats.range(f"F3:F{2 + n_rows}").api.NumberFormatLocal = "0,00"

    # --------------------------------------------------------
    # Foglio 3: Grafici
    # --------------------------------------------------------
    ws_graf = add_sheet(wb, "Grafici")

    # Titolo
    ws_graf.range("A1").value = "ANDAMENTO PREZZI — Ultimi 60 giorni"
    fmt_title(ws_graf.range("A1:F1"))
    ws_graf.range("A1:F1").merge()

    # Scrivi dati storici per il grafico
    ultimi_60 = prezzi.tail(60).copy()
    # Normalizza a base 100 per confronto visivo
    ultimi_60_norm = (ultimi_60 / ultimi_60.iloc[0]) * 100

    ws_graf.range("A3").options(index=True, header=True).value = ultimi_60_norm
    ws_graf.range("A3").api.NumberFormatLocal = "GG/MM/AAAA"

    last_row = 3 + len(ultimi_60_norm)

    # Crea grafico a linee
    top_pos = ws_graf.range(f"A{last_row + 2}").top
    chart = ws_graf.charts.add(
        left=10,
        top=top_pos,
        width=700,
        height=380
    )
    chart.chart_type = "line"
    chart.set_source_data(ws_graf.range(f"A3:F{last_row}"))
    chart.api[1].HasTitle = True
    chart.api[1].ChartTitle.Text = "Performance Comparata (base 100)"

    # Legenda
    chart.api[1].HasLegend = True
    chart.api[1].Legend.Position = -4107  # xlLegendPositionBottom

    autofit_all(wb)
    return wb


# Genera il report
print("Generazione report trimestrale...")
report = crea_report_bianchi(prezzi, stats)
print(f"Report creato con {len(report.sheets)} fogli: {[s.name for s in report.sheets]}")

### Struttura della funzione `crea_report_bianchi()`

Nota come la funzione e organizzata in **sezioni** chiaramente delimitate da commenti:

```python
def crea_report_bianchi(prezzi, stats):
    wb = create_workbook(visible=True)
    
    # --- Foglio 1: Copertina ---
    ws_cop = wb.sheets[0]
    # ... 20 righe ...
    
    # --- Foglio 2: Statistiche ---
    ws_stats = add_sheet(wb, "Statistiche")
    # ... 15 righe ...
    
    # --- Foglio 3: Grafici ---
    ws_graf = add_sheet(wb, "Grafici")
    # ... 20 righe ...
    
    return wb
```

Se la funzione diventa troppo lunga, ogni sezione puo diventare una funzione autonoma:  
`_crea_copertina(wb)`, `_crea_foglio_statistiche(wb, stats)`, ecc.

In [ ]:
# ============================================================
# ES12 - ESERCIZIO: Foglio Correlazione
# ============================================================
# Aggiungi un quarto foglio "Correlazione" al report esistente.
#
# Passi:
# 1. Calcola la matrice di correlazione dei rendimenti giornalieri
#    (usa prezzi.pct_change().dropna().corr())
#
# 2. Aggiungi il foglio "Correlazione" al workbook report
#    (usa add_sheet)
#
# 3. Scrivi un titolo nella riga 1, poi scrivi la matrice di correlazione
#    con indice e header a partire dalla riga 3
#
# 4. Colora le celle in base al valore della correlazione:
#    - corr > 0.8: verde scuro  (COLORS['correct_text'])  → fortemente correlati
#    - corr > 0.5: verde chiaro (COLORS['correct_bg'])    → correlati
#    - corr > 0.3: giallo       (COLORS['warning_bg'])    → debolmente correlati
#    - corr <= 0.3: rosso chiaro (COLORS['wrong_bg'])     → non correlati
#    Nota: la diagonale (correlazione = 1.0) puo essere ignorata o trattata a parte
#
# Suggerimento: itera con for i, titolo1 in enumerate(corr.index)
#               e for j, titolo2 in enumerate(corr.columns)

# ???

# Verifica
# fogli = [s.name for s in report.sheets]
# print(f"Fogli del report: {fogli}")
# assert "Correlazione" in fogli, "Foglio Correlazione mancante!"

In [ ]:
# ============================================================
# ES12 - SOLUZIONE
# ============================================================

# 1. Calcola matrice correlazione
rendimenti = prezzi.pct_change().dropna()
corr = rendimenti.corr().round(2)
print("Matrice di correlazione:")
print(corr)

# 2. Aggiungi foglio
ws_corr = add_sheet(report, "Correlazione")

# 3. Titolo
ws_corr.range("A1").value = "MATRICE DI CORRELAZIONE — Rendimenti Giornalieri"
fmt_title(ws_corr.range("A1:F1"))
ws_corr.range("A1:F1").merge()

# Scrivi la matrice con indice e header
ws_corr.range("A3").options(index=True, header=True).value = corr

# Formatta header riga 3
n_titoli = len(corr.columns)
fmt_header(ws_corr.range(f"A3:{chr(65 + n_titoli)}3"))

# 4. Colorazione heatmap
def corr_color(val):
    """Ritorna il colore di sfondo in base al valore di correlazione."""
    if val >= 1.0:
        return COLORS["locked_bg"]       # Diagonale: grigio neutro
    elif val > 0.8:
        return COLORS["correct_text"]    # Verde scuro
    elif val > 0.5:
        return COLORS["correct_bg"]      # Verde chiaro
    elif val > 0.3:
        return COLORS["warning_bg"]      # Giallo
    else:
        return COLORS["wrong_bg"]        # Rosso chiaro

for i, titolo1 in enumerate(corr.index):
    for j, titolo2 in enumerate(corr.columns):
        val = corr.loc[titolo1, titolo2]
        riga = 4 + i        # dati iniziano a riga 4
        col = 2 + j         # colonna B = 2, poi C, D...
        cella = ws_corr.range((riga, col))
        cella.api.NumberFormatLocal = "0,00"
        cella.color = corr_color(val)
        # Testo bianco su verde scuro per leggibilita
        if val > 0.8 and val < 1.0:
            cella.font.color = COLORS["header_text"]

autofit_all(report)

fogli = [s.name for s in report.sheets]
print(f"\nFogli del report: {fogli}")
print("Foglio Correlazione aggiunto con heatmap!")

---
## ES13: Esporta PDF

> *"I Bianchi vogliono il report via email. Serve un PDF. Uno solo, pulito, con il nostro logo nell'intestazione."*  
> — Dott. Ferretti

### Layout di stampa via API Excel

Prima di esportare in PDF, dobbiamo configurare il layout di stampa di ogni foglio.  
Excel espone queste impostazioni tramite `PageSetup`:

| Proprieta | Valore | Significato |
|-----------|--------|-------------|
| `Orientation` | 1 = Portrait, 2 = Landscape | Orientamento |
| `FitToPagesWide` | 1 | Adatta in larghezza a 1 pagina |
| `FitToPagesTall` | 1 | Adatta in altezza a 1 pagina |
| `CenterHorizontally` | True | Centra orizzontalmente |
| `LeftHeader` | stringa con codici | Intestazione sinistra |
| `CenterFooter` | `"&P di &N"` | Numero pagina / totale |

I **codici header/footer** di Excel: `&D` = data, `&P` = pagina corrente, `&N` = totale pagine,  
`&"Font,Bold"` = imposta font, `&L/&C/&R` = sinistra/centro/destra.

In [ ]:
# ============================================================
# ES13 - DEMO: Configura layout e esporta PDF
# ============================================================

# Configura il layout di stampa per ogni foglio
for ws in report.sheets:
    ps = ws.api.PageSetup
    ps.Orientation         = 2       # xlLandscape
    ps.Zoom                = False   # disabilita zoom fisso
    ps.FitToPagesWide      = 1       # adatta a 1 pagina in larghezza
    ps.FitToPagesTall      = 1       # adatta a 1 pagina in altezza
    ps.CenterHorizontally  = True
    ps.LeftMargin          = 36      # ~1.27 cm in punti
    ps.RightMargin         = 36
    ps.TopMargin           = 50
    ps.BottomMargin        = 50
    # Intestazione: nome societa a sinistra, data a destra
    ps.LeftHeader          = '&"Calibri,Bold"Turin Wealth Advisory'
    ps.RightHeader         = "&D"
    # Footer: pagina al centro
    ps.CenterFooter        = "Pagina &P di &N"
    ps.LeftFooter          = "Riservato — Famiglia Bianchi"

print("Layout di stampa configurato per tutti i fogli.")

# Esporta in PDF
pdf_path = os.path.join(OUTPUT_DIR, "Report_Bianchi_Completo.pdf")
report.api.ExportAsFixedFormat(
    Type=0,             # 0 = xlTypePDF
    Filename=pdf_path,
    Quality=0,          # 0 = xlQualityStandard
    IncludeDocProperties=True,
    IgnorePrintAreas=False,
    OpenAfterPublish=False
)
print(f"PDF esportato: {pdf_path}")

# Verifica che il file esista
if os.path.exists(pdf_path):
    size_kb = os.path.getsize(pdf_path) / 1024
    print(f"Dimensione PDF: {size_kb:.1f} KB")
else:
    print("[ERRORE] File PDF non creato!")

In [ ]:
# ============================================================
# ES13 - ESERCIZIO: Esporta solo alcuni fogli
# ============================================================
# Esporta in PDF solo i fogli "Statistiche" e "Grafici"
# (escludi Copertina e Correlazione).
#
# Strategia consigliata:
#   1. Nascondi temporaneamente i fogli che non vuoi nel PDF
#      (ws.api.Visible = 0  oppure  ws.api.Visible = False)
#   2. Esporta il workbook con ExportAsFixedFormat
#   3. Rendi visibili di nuovo i fogli nascosti
#      (ws.api.Visible = -1  oppure  ws.api.Visible = True)
#
# Il PDF deve chiamarsi "Report_Solo_Analisi.pdf"
#
# Suggerimento: usa un try/finally per garantire che i fogli
# vengano sempre resi visibili anche se ExportAsFixedFormat fallisce.

fogli_da_escludere = ["Copertina", "Correlazione"]

# ???

In [ ]:
# ============================================================
# ES13 - SOLUZIONE
# ============================================================

fogli_da_escludere = ["Copertina", "Correlazione"]
pdf_parziale = os.path.join(OUTPUT_DIR, "Report_Solo_Analisi.pdf")

# Nascondi fogli temporaneamente
nascosti = []
try:
    for ws in report.sheets:
        if ws.name in fogli_da_escludere:
            ws.api.Visible = 0   # xlSheetHidden
            nascosti.append(ws.name)
            print(f"  Nascosto temporaneamente: {ws.name}")

    # Esporta
    report.api.ExportAsFixedFormat(
        Type=0,
        Filename=pdf_parziale,
        Quality=0,
        IgnorePrintAreas=False,
        OpenAfterPublish=False
    )
    print(f"\nPDF parziale esportato: {pdf_parziale}")

finally:
    # Ripristina visibilita - eseguito SEMPRE anche in caso di errore
    for ws in report.sheets:
        if ws.name in nascosti:
            ws.api.Visible = -1  # xlSheetVisible
            print(f"  Ripristinato: {ws.name}")

print(f"\nFogli nel PDF: {[s.name for s in report.sheets if s.api.Visible == -1 and s.name not in fogli_da_escludere]}")
if os.path.exists(pdf_parziale):
    size_kb = os.path.getsize(pdf_parziale) / 1024
    print(f"Dimensione: {size_kb:.1f} KB")

---
## ES14: Il confronto finale

> *"Complimenti. Ma c'e una domanda che mi faccio sempre: i dati che usiamo per le esercitazioni  
> corrispondono alla realta? Confronta i dati simulati con quelli reali. Cosa hai imparato?"*  
> — Dott. Ferretti

### Simulazione vs Realta

I file `DATI_Turin_Wealth.xlsx` generati dagli script Python usano dati **simulati**:  
prezzi generati con `random.uniform()` e un seed fisso per la riproducibilita.  

I dati reali scaricati con yfinance mostrano caratteristiche che la simulazione semplice non cattura:

| Caratteristica | Simulazione semplice | Dati reali |
|----------------|---------------------|------------|
| Distribuzione rendimenti | Normale | Fat tails (code spesse) |
| Volatilita | Costante | Cluster (periodi calmi / turbolenti) |
| Autocorrelazione | Nulla | Lieve nelle varianze |
| Correlazione tra titoli | Programmata | Variabile nel tempo |
| Frequenza dati | Mensile | Giornaliera |


In [ ]:
# ============================================================
# ES14 - DEMO: Confronto simulazione vs realta
# ============================================================

path_dati = os.path.join(os.path.dirname(os.path.abspath("")), "output", "DATI_Turin_Wealth.xlsx")

try:
    # Carica dati simulati
    sim = pd.read_excel(path_dati, sheet_name="QUOTAZIONI_AZIONI", index_col=0)
    print("Dati simulati caricati da DATI_Turin_Wealth.xlsx")
    print(f"  Periodo simulato:  {sim.index[0]} -> {sim.index[-1]}")
    print(f"  Frequenza:         mensile ({len(sim)} osservazioni)")
    print(f"  Colonne:           {list(sim.columns)}")

    print(f"\nDati reali (yfinance):")
    print(f"  Periodo reale:     {prezzi.index[0].date()} -> {prezzi.index[-1].date()}")
    print(f"  Frequenza:         giornaliera ({len(prezzi)} osservazioni)")
    print(f"  Colonne:           {list(prezzi.columns)}")

    # Confronto volatilita
    print("\n--- Confronto volatilita annualizzata ---")
    rend_sim  = sim.pct_change().dropna()
    rend_real = prezzi.pct_change().dropna()

    # Volatilita mensile → annualizzata: *sqrt(12)
    vol_sim  = rend_sim.std()  * np.sqrt(12) * 100
    # Volatilita giornaliera → annualizzata: *sqrt(252)
    vol_real = rend_real.std() * np.sqrt(252) * 100

    print(f"  {'Titolo':<15} {'Simul. (%)':>12} {'Reale (%)':>12}")
    for az in AZIENDE:
        nome = az["nome"]
        v_s = vol_sim.get(nome, float("nan"))
        v_r = vol_real.get(nome, float("nan"))
        print(f"  {nome:<15} {v_s:>11.1f}% {v_r:>11.1f}%")

    print("\nNote:")
    print("  - I dati simulati usano random.uniform() con seed fisso")
    print("  - I dati reali mostrano cluster di volatilita (ARCH effects)")
    print("  - La simulazione semplice sottostima i rischi di coda (tail risk)")

except FileNotFoundError:
    print("[INFO] File DATI_Turin_Wealth.xlsx non trovato in output/.")
    print("  Per generarlo: cd scripts && python create_dati.py")
    print("  Procedi comunque con il confronto teorico nella cella successiva.")
except Exception as e:
    print(f"[ERRORE] {e}")

In [ ]:
# ============================================================
# ES14 - RIFLESSIONE GUIDATA
# ============================================================
# Rispondi alle domande qui sotto scrivendo le tue osservazioni.
# Non c'e una risposta giusta o sbagliata: e un esercizio di
# pensiero critico.

riflessioni = {
    "domanda_1": (
        "Perche i dati simulati sono utili per le esercitazioni "
        "anche se non corrispondono alla realta?"
    ),
    "risposta_1": (
        # Scrivi qui la tua risposta
        "???"
    ),

    "domanda_2": (
        "Quali limitazioni ha il modello random.uniform() "
        "rispetto ai modelli finanziari piu avanzati?"
    ),
    "risposta_2": (
        # Scrivi qui la tua risposta
        "???"
    ),

    "domanda_3": (
        "Se dovessi migliorare la simulazione, quale modello useresti? "
        "(es. GARCH, random walk con drift, Monte Carlo geometrico?)"
    ),
    "risposta_3": (
        # Scrivi qui la tua risposta
        "???"
    ),
}

print("Riflessioni ES14 — Simulazione vs Realta")
print("=" * 50)
for k, v in riflessioni.items():
    if k.startswith("domanda"):
        num = k.split("_")[1]
        print(f"\nDomanda {num}: {v}")
        print(f"Risposta:   {riflessioni.get('risposta_' + num, '???')}")

---
## Salva e chiudi

Esegui questa cella per salvare il report finale e chiudere Excel.

In [ ]:
# ============================================================
# SALVA IL REPORT FINALE
# ============================================================

output_path = os.path.join(OUTPUT_DIR, "Report_Trimestrale_Bianchi.xlsx")

try:
    report.save(output_path)
    print(f"Report salvato: {output_path}")
    if os.path.exists(output_path):
        size_kb = os.path.getsize(output_path) / 1024
        print(f"Dimensione file: {size_kb:.1f} KB")
except Exception as e:
    print(f"[ERRORE] Salvataggio fallito: {e}")
    print("Assicurati che il file non sia aperto in un altro programma.")
    raise

# Chiudi Excel
try:
    report.close()
    report.app.quit()
    print("Excel chiuso correttamente.")
except Exception:
    print("[INFO] Excel gia chiuso o istanza non trovata.")

print(f"\nFile generati in {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    if f.endswith((".xlsx", ".pdf")):
        path_f = os.path.join(OUTPUT_DIR, f)
        print(f"  {f}  ({os.path.getsize(path_f)/1024:.1f} KB)")

---
## Riepilogo del Blocco 3

In questo blocco hai costruito una pipeline completa di automazione Excel con Python.

### Competenze acquisite

| Esercizio | Concetto chiave | Funzione/Strumento |
|-----------|-----------------|--------------------|
| ES10 | Organizzazione in funzioni | `def`, parametri, return |
| ES11 | Protezione fogli e celle | `api.Locked`, `FormulaHidden`, `protect_sheet()` |
| ES12 | Report multi-foglio | `add_sheet()`, `write_table()`, grafici xlwings |
| ES13 | Export PDF | `PageSetup`, `ExportAsFixedFormat` |
| ES14 | Pensiero critico | Simulazione vs dati reali |

### Pattern da ricordare

```python
# Pipeline standard Turin Wealth:
prezzi = scarica_dati_aziende()        # 1. Scarica
stats  = calcola_statistiche(prezzi)   # 2. Elabora
wb     = crea_report_bianchi(prezzi, stats)  # 3. Crea Excel
wb.api.ExportAsFixedFormat(0, pdf_path)      # 4. Esporta PDF
wb.save(xlsx_path)                     # 5. Salva
wb.close(); wb.app.quit()              # 6. Chiudi
```

### Prossimo blocco

**Blocco 4**: Analisi fondamentale — P/E, DCF, confronto multipli tra aziende.  
Alessandro Bianchi vuole investire €200k in azioni singole. Dovrai giustificare ogni scelta con i numeri.

---
*Turin Wealth Advisory — Corso universitario di Excel, Facolta di Economia*